In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from groq import Groq
import os 

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
model = os.getenv("GROQ_MODEL")

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=0.5):
    params = {
        "model": model,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    response = client.chat.completions.create(**params)
    return response.choices[0].message

In [21]:
def to_json(text):
    return json.loads(text.content.strip("```").strip("json"))

In [23]:
# Function to generate a new dataset
import json


def generate_dataset():

    system_prompt = """You are an evaluation dataset generator.
        Your task is to generate dataset samples for evaluating prompts related to AWS tasks.
        You MUST always output a valid JSON array of objects following this exact schema:

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex",
                "solution_criteria": "Key criteria for evaluating the solution"
            },
            ...additional
        ]
        ```
        Focus on tasks that do not require writing much code.
        Do not include any introductory or concluding prose.
        """
    user_prompt = """Focus on AWS-related tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
Please generate 3 objects."""

    messages = []
    add_user_message(messages, user_prompt)
    add_assistant_message(messages, system_prompt)
    text = chat(messages)
    result = text
    return to_json(result)

In [24]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_system_prompt = """
    You are an expert AWS code reviewer. Your task is to evaluate AI-generated solutions.

    Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""
    
    eval_user_prompt = f"""

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output["content"]}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>
    """

    messages = []
    add_user_message(messages, eval_user_prompt)
    add_assistant_message(messages, eval_system_prompt)
    eval_text = chat(messages)
    return to_json(eval_text)

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

In [ ]:
output = "^arn:aws:lambda:[a-z]{2}-[a-z]+-\d+:\d{12}:function:[a-zA-Z0-9_-]+$"
case_test = dataset[2]

In [ ]:
result = grade_by_model(case_test, output)

In [ ]:
print(result)

{'strengths': ['Uses start (^) and end ($) anchors to ensure the entire string is matched', 'Correctly validates the 12‑digit account ID and allowed characters in the function name', 'Accurately captures the typical AWS region pattern (two letters‑dash‑letters‑dash‑digits)'], 'weaknesses': ['Region part is overly permissive – it allows any length of letters after the first dash, not just the known region identifiers', 'Does not explicitly prevent leading zeros in the numeric region suffix (e.g., "us-east-01")'], 'reasoning': 'The regex meets the core requirements: full‑string matching, correct account‑ID length, and allowed function‑name characters. It correctly models the common region format, though it could be tightened to reject impossible region strings. Overall it is a solid solution with minor over‑permissiveness.', 'score': 9}


In [ ]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages)
    return output

In [ ]:
result = run_prompt(case_test)

In [ ]:
print("result: ",result)

result:  ChatCompletionMessage(content='^arn:aws:lambda:[a-z]{2}-[a-z]+-\\d+:\\d{12}:function:[A-Za-z0-9_-]+$', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to output just a regex, no comments. Provide regex that matches ARN pattern. Region: two-letter code? Actually AWS region like us-east-1: two letters then dash then word? The description: two‑letter code followed by a dash and a number (e.g., `us-east-1`). Actually "two-letter code followed by a dash and a number" maybe means like "us-east-1": two letters "us", dash, then something? But typical region pattern: [a-z]{2}-[a-z]+-\\d+. We\'ll follow that. Account-id: \\d{12}. Function name: [A-Za-z0-9_-]+. Full pattern: ^arn:aws:lambda:[a-z]{2}-[a-z]+-\\d+:\\d{12}:function:[A-Za-z0-9_-]+$.\n\nReturn just regex. Probably as raw string: `^arn:aws:lambda:[a-z]{2}-[a-z]+-\\d+:\\d{12}:function:[A-Za-z0-9_-]+$`. Provide without code fences? The instruction: Respond only with Python, JSON, or

In [ ]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [ ]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    print(output.content)
    model_grade = grade_by_model(test_case, output.content)
    print("Model grade:", model_grade)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [ ]:
case_test ={
    "task": "Write a regular expression that matches valid AWS Lambda function ARNs of the form `arn:aws:lambda:<region>:<account-id>:function:<function-name>` where `<region>` is a two\u2011letter code followed by a dash and a number (e.g., `us-east-1`), `<account-id>` is a 12\u2011digit number, and `<function-name>` consists of alphanumeric characters, hyphens, or underscores.",
    "format": "regex",
    "solution_criteria": "The regex should be a single pattern string (no surrounding delimiters) that captures the entire ARN and validates each component as described. It must reject ARNs with missing parts, wrong region format, non\u2011numeric account IDs, or illegal characters in the function name."
  }

In [ ]:
output = run_test_case(case_test)

^arn:aws:lambda:[a-z]{2}(?:-[a-z]+)*-\d+:\d{12}:function:[A-Za-z0-9_-]+$


In [ ]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

```python
import boto3

def list_s3_keys(bucket_name: str, prefix: str) -> list:
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
    keys = []
    for page in page_iterator:
        contents = page.get('Contents', [])
        for obj in contents:
            keys.append(obj['Key'])
    return keys
```
test_case:  {'task': 'Write a Python function that takes an AWS S3 bucket name and a prefix string, and returns a list of all object keys in that bucket that start with the given prefix. Assume the function will be run with appropriate AWS credentials and the `boto3` library is available.', 'format': 'python', 'solution_criteria': "The solution must define a single function (e.g., `list_objects(bucket, prefix)`) that uses `boto3.client('s3')`, calls `list_objects_v2` (handling pagination if needed), filters keys by the prefix, and returns a plain Python list of strings. No additional 

AttributeError: 'str' object has no attribute 'content'

In [ ]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n    \"lambda_function\": {\n        \"FunctionName\": \"database-connection-handler\",\n        \"Runtime\": \"python3.9\",\n        \"Environment\": {\n            \"Variables\": {\n                \"DB_HOST\": \"your-database-hostname.rds.amazonaws.com\",\n                \"DB_PORT\": \"5432\",\n                \"DB_NAME\": \"myappdatabase\",\n                \"DB_USERNAME\": \"dbadminuser\", \n                \"DB_PASSWORD\": \"{{resolve:secretsmanager:DatabaseCredentials:SecretString:password}}\",\n                \"DB_SSL_MODE\": \"require\"\n            }\n        },\n        \"Timeout\": 30,\n        \"MemorySize\": 256\n    }\n}\n",
    "test_case": {
      "task": "Create a JSON configuration for an AWS Lambda function that sets environment variables for database connection",
      "format": "json"
    },
    "score": 8.5,
    "reasoning": "The solution demonstrates good foundational practices for Lambda environment configuration with database connect